In [ ]:
!pip install transformers datasets torchmetrics

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

In [ ]:
REPO_ID = "VasilisAsim/wav2vec-finetuned-TESS"
NUM_SAMPLES = 150
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"].shuffle(seed=42).select(range(NUM_SAMPLES))

In [ ]:
#Ετικέτες TESS
labels = sorted(set(dataset["train"]["emotion"]))
label2id_fixed = {str(label): i for i, label in enumerate(labels)}
id2label_fixed = {i: str(label) for i, label in enumerate(labels)}
print("Κλάσεις:", labels)

In [ ]:
def extract_embeddings(model, feature_extractor, dataset, encoder_attr="wav2vec2"):
    model.eval()
    model.to(device)

    encoder = getattr(model, encoder_attr)
    num_layers = model.config.num_hidden_layers + 1
    layer_buffers = [[] for _ in range(num_layers)]

    for sample in dataset:
        audio = sample["audio"]
        inputs = feature_extractor(
            audio["array"], sampling_rate=audio["sampling_rate"],
            max_length=6*16000, truncation=True, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = encoder(**inputs, output_hidden_states=True)

        for layer_idx, hidden_state in enumerate(outputs.hidden_states):
            pooled = hidden_state.squeeze(0).mean(dim=0).cpu().numpy()
            layer_buffers[layer_idx].append(pooled)

    return {layer_idx: np.stack(buf) for layer_idx, buf in enumerate(layer_buffers)}

In [ ]:
def linear_CKA(X, Y):
    """Γραμμική Centered Kernel Alignment μεταξύ δύο πινάκων αναπαραστάσεων
    (δείγματα × hidden_dim), για το ίδιο layer σε δύο διαφορετικά checkpoints."""
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)

    hsic = np.linalg.norm(X.T @ Y, ord="fro") ** 2
    var1 = np.linalg.norm(X.T @ X, ord="fro")
    var2 = np.linalg.norm(Y.T @ Y, ord="fro")

    return hsic / (var1 * var2)

In [ ]:
checkpoints_all = [
    {"name": "finetuned", "repo_id": "VasilisAsim/wav2vec-finetuned-TESS", "revision": None},
]

In [ ]:
print("Εξαγωγή embeddings: pretrained...")
feature_extractor_or = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model_or = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=len(label2id_fixed), label2id=label2id_fixed, id2label=id2label_fixed,
    ignore_mismatched_sizes=True,
)
emb_pre = extract_embeddings(model_or, feature_extractor_or, test_dataset, encoder_attr="wav2vec2")

In [ ]:
emb_all = {}
for ckpt in checkpoints_all:
    print(f"Εξαγωγή embeddings: {ckpt['name']} ({ckpt['repo_id']})...")
    fe_ckpt = AutoFeatureExtractor.from_pretrained(ckpt["repo_id"], revision=ckpt["revision"])
    model_ckpt = AutoModelForAudioClassification.from_pretrained(ckpt["repo_id"], revision=ckpt["revision"])
    emb_all[ckpt["name"]] = extract_embeddings(model_ckpt, fe_ckpt, test_dataset, encoder_attr="wav2vec2")

In [ ]:
#Υπολογισμός CKA ανά layer
num_layers = len(emb_pre)
cka_rows = []
for ckpt_name, emb_ckpt in emb_all.items():
    for l in range(num_layers):
        score = linear_CKA(emb_pre[l], emb_ckpt[l])
        cka_rows.append({"checkpoint": ckpt_name, "layer": l, "cka": score})

cka_results = pd.DataFrame(cka_rows)
print(cka_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sub = cka_results[cka_results["checkpoint"] == "finetuned"]
ax.plot(sub["layer"], sub["cka"], marker="o", label="finetuned", color="tab:green")

ax.set_xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
ax.set_ylabel("CKA similarity (Pretrained vs Finetuned)")
ax.set_title("Layer-wise CKA: pretrained vs finetuned")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()